In [68]:
import pandas as pd
from pathlib import Path


# -------------------------------
# 1 Read the data
# -------------------------------

# Path() is used to create a file path object
# This points to data/raw/AAPL_dirty_practice_comma.csv inside the project directory
data_path = Path("data/raw/AAPL_dirty_practice_comma.csv")

# pd.read_csv()
# Used to read a CSV file and return a pandas DataFrame
df = pd.read_csv(data_path)



# -------------------------------
# 2 Standardize column names
# -------------------------------

# df.columns is an Index object that stores all column names
# .str.strip() removes spaces at the beginning and end of each string
# .str.lower() converts all column names to lowercase
# This helps avoid issues such as:
#   " Date "
#   "HIGH "
#   "Close Price"
# inconsistent capitalization and extra spaces in column names
df.columns = df.columns.str.strip().str.lower()



# -------------------------------
# 3 Rename columns
# -------------------------------

# rename() is used to modify column names
# columns={} specifies which columns should be renamed
# key   → original column name
# value → new column name
# Here we only rename the columns with spaces to cleaner names
df = df.rename(columns={
    "open price": "open",
    "close price": "close"
})


# Check whether the column names are correct
print(df.columns)



# -------------------------------
# 4 Clean the date text
# -------------------------------

# astype(str)
# Convert the entire date column to string
# This is needed because the .str methods will be used later for string processing
df["date"] = df["date"].astype(str)


# str.replace(old, new, regex=False)
# old  → the string to be replaced
# new  → the replacement string
# regex=False → match as a normal string, not as a regular expression
# Remove "date:" from values such as "date: 1980-12-18"
df["date"] = df["date"].str.replace("date:", "", regex=False)


# Remove double quotation marks "
# For example, "Dec 17, 1980"
df["date"] = df["date"].str.replace('"', "", regex=False)


# str.strip()
# Remove spaces at both ends of the string
# For example:
# " 1980-12-12 " → "1980-12-12"
df["date"] = df["date"].str.strip()



# -------------------------------
# 5 Convert to datetime type
# -------------------------------

# pd.to_datetime()
# Convert strings to pandas datetime type

df["date"] = pd.to_datetime(

    df["date"],

    # errors="coerce"
    # If a value cannot be parsed as a date, convert it to NaT (Not a Time)
    # For example:
    #   "not available"
    #   "summary row"
    #   "1981/02/30"
    # all of them will become NaT
    errors="coerce",

    # dayfirst=True
    # When a value is in the format 23/12/1980
    # parse it as day/month/year
    # otherwise, the default would be month/day/year
    dayfirst=True
)



# -------------------------------
# 6 Inspect dates that could not be parsed
# -------------------------------

# isna()
# Check whether values are missing
# In datetime columns, NaT is also treated as a missing value

# df[condition]
# used to filter rows that satisfy the condition

bad_rows = df[df["date"].isna()]


# Display these problematic rows
bad_rows

Index(['date', 'open', 'high', 'low', 'close'], dtype='object')


,date,open,high,low,close
1,NaT,0.094108 USD,0.094108 USD,0.093678 USD,0.093678 USD
2,NaT,price=0.087232,price=0.087232,price=0.086802,price=0.086802
3,NaT,0.088951 dollars,0.089381 dollars,0.088951 dollars,NaN
4,NaT,approx. $0.091530,approx. $0.091959,approx. $0.091530,approx. $0.091530
5,NaT,0.097116,0.097545,0.097116,0.097116
6,NaT,$0.101842,$0.102273,$0.101842,$0.101842
7,NaT,?,0.106570 USD,0.106140 USD,0.106140 USD
8,NaT,generated for practice,NaN,NaN,NaN
9,NaT,price=0.111726,price=0.112156,price=0.111726,price=0.111726
10,NaT,0.122039 dollars,missing,0.122039 dollars,0.122039 dollars


#pd.read_csv("data.csv", header=None)
# usecols=["Date", "Close"] reads only selected columns
# Read only selected columns

In [69]:
df = df.dropna(subset=["date"])# subset=["date"] checks only the date column; dropna() removes missing values

,Date,Open Price,HIGH,low,Close Price
0,1980-12-12,$0.098834,$0.099264,$0.098834,$0.098834
1,15/12/1980,0.094108 USD,0.094108 USD,0.093678 USD,0.093678 USD
2,12-16-1980,price=0.087232,price=0.087232,price=0.086802,price=0.086802
3,"Dec 17, 1980",0.088951 dollars,0.089381 dollars,0.088951 dollars,NaN
4,date: 1980-12-18,approx. $0.091530,approx. $0.091959,approx. $0.091530,approx. $0.091530


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44 entries, 0 to 43
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0    Date         43 non-null     object
 1   Open Price    44 non-null     object
 2   HIGH          42 non-null     object
 3    low          40 non-null     object
 4   Close Price   41 non-null     object
dtypes: object(5)
memory usage: 1.8+ KB


Index([' Date ', 'Open Price', 'HIGH ', ' low', 'Close Price '], dtype='object')


,date,open,high,low,close
1,NaT,0.094108 USD,0.094108 USD,0.093678 USD,0.093678 USD
2,NaT,price=0.087232,price=0.087232,price=0.086802,price=0.086802
3,NaT,0.088951 dollars,0.089381 dollars,0.088951 dollars,NaN
4,NaT,approx. $0.091530,approx. $0.091959,approx. $0.091530,approx. $0.091530
5,NaT,0.097116,0.097545,0.097116,0.097116
6,NaT,$0.101842,$0.102273,$0.101842,$0.101842
7,NaT,?,0.106570 USD,0.106140 USD,0.106140 USD
8,NaT,generated for practice,NaN,NaN,NaN
9,NaT,price=0.111726,price=0.112156,price=0.111726,price=0.111726
10,NaT,0.122039 dollars,missing,0.122039 dollars,0.122039 dollars


In [70]:
# Price columns
price_cols = ["open", "high", "low", "close"]

for col in price_cols:
    
    # Convert to string
    df[col] = df[col].astype(str)
    
    # Remove text markers
    df[col] = df[col].str.replace("$", "", regex=False)
    df[col] = df[col].str.replace("USD", "", regex=False)
    df[col] = df[col].str.replace("price=", "", regex=False)
    df[col] = df[col].str.replace("approx.", "", regex=False)
    df[col] = df[col].str.replace("dollars", "", regex=False)
    df[col] = df[col].str.replace("close=", "", regex=False)
    
    # Remove spaces
    df[col] = df[col].str.strip()

In [73]:
for col in price_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [74]:
for col in price_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [76]:
df = df[(df["close"] > 0) & (df["close"] < 100)]# remove outliers
df = df.dropna(subset=price_cols)# remove missing prices

In [77]:
#final check
df.info()
df.describe()
df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 4 entries, 0 to 39
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    4 non-null      datetime64[ns]
 1   open    4 non-null      float64       
 2   high    4 non-null      float64       
 3   low     4 non-null      float64       
 4   close   4 non-null      float64       
dtypes: datetime64[ns](1), float64(4)
memory usage: 192.0 bytes


,date,open,high,low,close
0,1980-12-12,0.098834,0.099264,0.098834,0.098834
20,1981-09-01,0.000000,0.110007,0.109577,0.109577
37,1981-02-02,0.091959,0.091959,0.091530,0.091530
39,1981-04-02,0.098405,0.098834,0.098405,0.098405
